# TXED → robust EQViT magnitude estimation: training, validation, and verification

This is the **main end-to-end training notebook** for EQViT-torch.

It is designed to answer four practical questions:

1. Can the local `TXED_20231111.h5` / `ID_20231111.npy` files be read correctly?
2. Can we train a **single-station, P-centered EQViT magnitude estimator** without event leakage?
3. Does the trained model generalize on completely held-out TexNet earthquakes?
4. Where does it fail as a function of **magnitude, SNR, station, and P-pick error**?

### Scientific design

* TXED waveform: 6000 × 3 at 100 Hz.
* Magnitude input: 3000 samples = **1 s before P + 29 s after P**.
* **No trace-by-trace amplitude normalization** is used for magnitude estimation.
* Splitting is by **event ID**, not waveform ID, so recordings of the same earthquake cannot occur in both training and testing.
* Training supports P-pick jitter so the magnitude model is not unrealistically dependent on a perfect manual P pick.
* A magnitude-balanced sampler is optional because TXED contains many more small than large earthquakes.
* Final verification includes waveform QC, learning curves, predicted-vs-catalog magnitude, residuals, magnitude/SNR/station stratification, large-event metrics, and P-pick perturbation stress tests.

> Start with `QUICK_RUN=True`. After the entire notebook runs correctly on your machine, set it to `False`.

In [ ]:
from pathlib import Path
import os, random, time, math, json
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from eqvit_torch.models import EQViTMagnitude

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else (
    "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu"
)
print("PyTorch:", torch.__version__)
print("Device :", DEVICE)

## 1. Point the notebook to your local TXED

The official TXED examples use `TXED_20231111.h5` and `ID_20231111.npy`. Change only `TXED_DIR` below.

The notebook validates the files before doing any training.

In [ ]:
# EDIT THIS PATH ONLY
TXED_DIR = Path("/Users/chenyk/DATALIB/TXED")   # <-- change if needed

H5_PATH = TXED_DIR / "TXED_20231111.h5"
ID_PATH = TXED_DIR / "ID_20231111.npy"

OUTPUT_DIR = Path("./outputs/txed_eqvit_magnitude")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

QUICK_RUN = True       # True: pipeline/debug run; False: full experiment
MAX_EPOCHS = 8 if QUICK_RUN else 200
PATIENCE = 4 if QUICK_RUN else 25
BATCH_SIZE = 32 if QUICK_RUN else 128
NUM_WORKERS = 0        # keep 0 first; increase later if desired
LR = 1e-3
WEIGHT_DECAY = 1e-5
P_JITTER_SAMPLES = 20  # ±0.20 s during training; set 0 for exact manual P
BALANCED_SAMPLING = True

assert H5_PATH.exists(), f"Missing {H5_PATH}"
assert ID_PATH.exists(), f"Missing {ID_PATH}"
print("H5 :", H5_PATH, H5_PATH.stat().st_size/1024**3, "GB")
print("IDs:", ID_PATH)

## 2. Audit the TXED structure before training

This cell checks the actual HDF5 record layout and the metadata used by the model. It intentionally fails early if required attributes are absent.

In [ ]:
all_ids = np.load(ID_PATH, allow_pickle=True).astype(str)
signal_ids = np.array([x for x in all_ids if x.split("_")[-1] == "EV"])
noise_ids  = np.array([x for x in all_ids if x.split("_")[-1] == "NO"])

print(f"All records   : {len(all_ids):,}")
print(f"Signal records: {len(signal_ids):,}")
print(f"Noise records : {len(noise_ids):,}")

required = {"p_arrival_sample", "magnitude", "snr_db"}
with h5py.File(H5_PATH, "r") as f:
    wid = signal_ids[0]
    g = f[wid]
    print("\nExample ID:", wid)
    print("data shape:", g["data"].shape, "dtype:", g["data"].dtype)
    print("attributes:", sorted(g.attrs.keys()))
    missing = required - set(g.attrs.keys())
    assert not missing, f"Missing required TXED attributes: {missing}"
    assert g["data"].shape == (6000, 3), "Expected TXED waveform shape (6000,3)"
    print("P sample:", g.attrs["p_arrival_sample"])
    print("Magnitude:", g.attrs["magnitude"])
    print("SNR dB:", g.attrs["snr_db"])

## 3. Build a compact metadata index

The HDF5 waveform arrays are not loaded here; only metadata are scanned. The resulting table is useful for QC, splitting, balanced sampling, and later diagnostic plots.

For a quick test, only a subset is indexed. Full training uses all signal records.

In [ ]:
def _safe_float(x, default=np.nan):
    try: return float(x)
    except Exception: return default

scan_ids = signal_ids[:min(len(signal_ids), 25000 if QUICK_RUN else len(signal_ids))]
rows = []
t0 = time.time()
with h5py.File(H5_PATH, "r") as f:
    for k, wid in enumerate(scan_ids):
        a = f[wid].attrs
        snr = np.asarray(a.get("snr_db", [np.nan]*3), dtype=float)
        rows.append({
            "id": wid,
            "event_id": wid.split("_")[0],
            "station": wid.split("_")[1],
            "magnitude": _safe_float(a.get("magnitude")),
            "p_sample": _safe_float(a.get("p_arrival_sample")),
            "snr_mean": float(np.nanmean(snr)),
            "ev_latitude": _safe_float(a.get("ev_latitude")),
            "ev_longitude": _safe_float(a.get("ev_longitude")),
            "ev_depth_m": _safe_float(a.get("ev_depth")),
        })
meta = pd.DataFrame(rows)
meta = meta.replace([np.inf, -np.inf], np.nan)
meta = meta.dropna(subset=["magnitude", "p_sample"]).reset_index(drop=True)

print(f"Indexed {len(meta):,} records in {time.time()-t0:.1f} s")
display(meta.head())
display(meta[["magnitude","p_sample","snr_mean","ev_depth_m"]].describe())
print("Unique earthquakes:", meta.event_id.nunique())
print("Unique stations   :", meta.station.nunique())

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
ax.hist(meta["magnitude"], bins=60)
ax.set(xlabel="Catalog magnitude", ylabel="Waveform count",
       title="TXED magnitude distribution")
ax.grid(alpha=.2)
plt.show()

fig, ax = plt.subplots(figsize=(8,4))
ax.hist(meta["snr_mean"].dropna(), bins=60)
ax.set(xlabel="Mean 3-C SNR (dB)", ylabel="Waveform count",
       title="TXED SNR distribution")
ax.grid(alpha=.2)
plt.show()

## 4. Visual QC: raw TXED waveforms and the 30-s magnitude window

The magnitude window is **not independently normalized**. We only remove the per-component mean. This preserves amplitude information needed for magnitude estimation.

In [ ]:
def extract_window(x, p, n=3000, pre=100):
    p = int(round(p))
    start = p - pre
    out = np.zeros((n, 3), dtype=np.float32)
    a, b = max(0, start), min(len(x), start+n)
    if b > a:
        out[a-start:b-start] = x[a:b]
    return out

rng = np.random.default_rng(SEED)
show_ids = rng.choice(meta.id.values, size=min(4, len(meta)), replace=False)

with h5py.File(H5_PATH, "r") as f:
    for wid in show_ids:
        g = f[wid]
        x = np.asarray(g["data"], np.float32)
        p = int(g.attrs["p_arrival_sample"])
        m = float(g.attrs["magnitude"])
        w = extract_window(x, p)
        w = w - w.mean(axis=0, keepdims=True)

        t = np.arange(len(w))/100.0 - 1.0
        fig, ax = plt.subplots(figsize=(10,4))
        scale = np.max(np.abs(w), axis=0)
        scale[scale == 0] = 1
        for j, lab in enumerate(["Z","N/1","E/2"]):
            ax.plot(t, w[:,j]/scale[j] + 2*j, lw=.7, label=lab)
        ax.axvline(0, ls="--", lw=1)
        ax.set(xlabel="Time relative to P (s)", ylabel="Normalized for display only",
               title=f"{wid} | catalog M={m:.2f}")
        ax.legend(ncol=3)
        ax.grid(alpha=.15)
        plt.show()

## 5. Event-disjoint train/validation/test split

This is critical. TXED has multiple station records for many earthquakes. A waveform-random split can leak the same physical earthquake into both training and testing.

We therefore split **unique TexNet event IDs** first and only then assign waveforms.

In [ ]:
events = np.array(sorted(meta.event_id.unique()))
rng = np.random.default_rng(SEED)
rng.shuffle(events)

n = len(events)
n_train = int(0.80*n)
n_val = int(0.10*n)
train_events = set(events[:n_train])
val_events   = set(events[n_train:n_train+n_val])
test_events  = set(events[n_train+n_val:])

train_df = meta[meta.event_id.isin(train_events)].copy()
val_df   = meta[meta.event_id.isin(val_events)].copy()
test_df  = meta[meta.event_id.isin(test_events)].copy()

assert set(train_df.event_id).isdisjoint(val_df.event_id)
assert set(train_df.event_id).isdisjoint(test_df.event_id)
assert set(val_df.event_id).isdisjoint(test_df.event_id)

print("waveforms:", len(train_df), len(val_df), len(test_df))
print("events   :", train_df.event_id.nunique(), val_df.event_id.nunique(), test_df.event_id.nunique())

for name, d in [("train",train_df),("validation",val_df),("test",test_df)]:
    print(f"{name:10s}: M=[{d.magnitude.min():.2f}, {d.magnitude.max():.2f}], "
          f"median={d.magnitude.median():.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
bins = np.linspace(meta.magnitude.min(), meta.magnitude.max(), 50)
ax.hist(train_df.magnitude, bins=bins, density=True, histtype="step", label="train")
ax.hist(val_df.magnitude, bins=bins, density=True, histtype="step", label="validation")
ax.hist(test_df.magnitude, bins=bins, density=True, histtype="step", label="test")
ax.set(xlabel="Catalog magnitude", ylabel="Density",
       title="Event-disjoint split: magnitude distributions")
ax.legend()
ax.grid(alpha=.2)
plt.show()

## 6. Dataset: amplitude-preserving, P-centered, with realistic pick jitter

`jitter_samples` is used only for training. Validation and test data remain deterministic.

The jitter is important for the eventual EQCCT-torch → EQViT-torch workflow: the magnitude model should not collapse when the automatic P pick is tens of samples away from the analyst pick.

In [ ]:
class TXEDMagnitudeDataset(Dataset):
    def __init__(self, h5_path, frame, jitter_samples=0, seed=2026):
        self.h5_path = str(h5_path)
        self.frame = frame.reset_index(drop=True)
        self.jitter_samples = int(jitter_samples)
        self.seed = int(seed)
        self._h5 = None

    def __len__(self):
        return len(self.frame)

    def _file(self):
        if self._h5 is None:
            self._h5 = h5py.File(self.h5_path, "r")
        return self._h5

    def __getitem__(self, i):
        r = self.frame.iloc[i]
        g = self._file()[r.id]
        x = np.asarray(g["data"], np.float32)
        p0 = int(round(float(g.attrs["p_arrival_sample"])))

        if self.jitter_samples:
            # deterministic per item/worker epoch-independent jitter is enough for a robust baseline;
            # replace by dynamic augmentation later if desired.
            rr = np.random.default_rng(self.seed + i)
            dp = int(rr.integers(-self.jitter_samples, self.jitter_samples+1))
        else:
            dp = 0

        w = extract_window(x, p0 + dp, n=3000, pre=100)
        w = w - w.mean(axis=0, keepdims=True)  # preserve relative/raw amplitude

        y = np.float32(g.attrs["magnitude"])
        return (
            torch.from_numpy(np.ascontiguousarray(w)),
            torch.tensor(y),
            {
                "id": r.id,
                "event_id": r.event_id,
                "station": r.station,
                "snr_mean": np.float32(r.snr_mean),
                "p_jitter": dp,
            },
        )

train_ds = TXEDMagnitudeDataset(H5_PATH, train_df, jitter_samples=P_JITTER_SAMPLES, seed=SEED)
val_ds   = TXEDMagnitudeDataset(H5_PATH, val_df, jitter_samples=0, seed=SEED)
test_ds  = TXEDMagnitudeDataset(H5_PATH, test_df, jitter_samples=0, seed=SEED)

x0, y0, m0 = train_ds[0]
print(x0.shape, y0.item(), m0)
assert x0.shape == (3000,3)
assert torch.isfinite(x0).all()

## 7. Optional magnitude-balanced sampling

A model optimized on the raw TXED distribution can minimize average loss by focusing heavily on abundant small earthquakes. For EEW, performance on M≥3 and especially the largest Texas events matters disproportionately.

The sampler below approximately equalizes **magnitude-bin exposure** while leaving the validation and test distributions untouched.

In [ ]:
def magnitude_weights(mags, width=0.5):
    mags = np.asarray(mags, float)
    lo = math.floor(np.nanmin(mags)/width)*width
    hi = math.ceil(np.nanmax(mags)/width)*width + width
    edges = np.arange(lo, hi+1e-9, width)
    idx = np.clip(np.digitize(mags, edges)-1, 0, len(edges)-2)
    counts = np.bincount(idx, minlength=len(edges)-1)
    w = 1.0 / np.maximum(counts[idx], 1)
    w /= w.mean()
    return w

if BALANCED_SAMPLING:
    weights = magnitude_weights(train_df.magnitude.values, width=0.5)
    sampler = WeightedRandomSampler(
        torch.as_tensor(weights, dtype=torch.double),
        num_samples=len(weights),
        replacement=True,
    )
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=NUM_WORKERS, pin_memory=(DEVICE=="cuda"))
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=(DEVICE=="cuda"))

val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE*2, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=(DEVICE=="cuda"))
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE*2, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=(DEVICE=="cuda"))

print("batches:", len(train_loader), len(val_loader), len(test_loader))

## 8. Build the EQViT magnitude model

The model follows the EQViT magnitude branch: convolutional feature extraction followed by ViT/attention and scalar magnitude regression.

We use AdamW and Smooth-L1 (Huber) loss for a robust TXED baseline. Validation selection is still based on **MAE**, so the reported model is directly interpretable.

In [ ]:
model = EQViTMagnitude().to(DEVICE)
npar = sum(p.numel() for p in model.parameters())
print(f"Parameters: {npar:,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=np.sqrt(0.1), patience=10
)
criterion = nn.SmoothL1Loss(beta=0.25)

def run_epoch(model, loader, training=False):
    model.train(training)
    ys, ps = [], []
    total_loss, n = 0.0, 0

    for x, y, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            pred = model(x).reshape(-1)
            loss = criterion(pred, y.reshape(-1))
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()

        total_loss += float(loss.detach()) * len(x)
        n += len(x)
        ys.append(y.detach().cpu().numpy())
        ps.append(pred.detach().cpu().numpy())

    y = np.concatenate(ys)
    p = np.concatenate(ps)
    e = p-y
    return {
        "loss": total_loss/max(n,1),
        "mae": float(np.mean(np.abs(e))),
        "rmse": float(np.sqrt(np.mean(e**2))),
        "bias": float(np.mean(e)),
        "std": float(np.std(e)),
        "y": y, "pred": p,
    }

## 9. Train with checkpointing and early stopping

The best checkpoint is selected only from validation MAE. The test set is never used for model selection.

In [ ]:
BEST = OUTPUT_DIR / "txed_eqvit_magnitude_best.pt"
history = []
best_mae = np.inf
bad_epochs = 0

for epoch in range(1, MAX_EPOCHS+1):
    t0 = time.time()
    tr = run_epoch(model, train_loader, training=True)
    va = run_epoch(model, val_loader, training=False)
    scheduler.step(va["mae"])

    row = {
        "epoch": epoch,
        "train_loss": tr["loss"],
        "train_mae": tr["mae"],
        "val_loss": va["loss"],
        "val_mae": va["mae"],
        "val_rmse": va["rmse"],
        "lr": optimizer.param_groups[0]["lr"],
        "seconds": time.time()-t0,
    }
    history.append(row)
    print(f"E{epoch:03d} train MAE={tr['mae']:.3f} | "
          f"val MAE={va['mae']:.3f} RMSE={va['rmse']:.3f} | "
          f"lr={row['lr']:.2e} | {row['seconds']:.1f}s")

    if va["mae"] < best_mae - 1e-4:
        best_mae = va["mae"]
        bad_epochs = 0
        torch.save({
            "state_dict": model.state_dict(),
            "epoch": epoch,
            "val_mae": best_mae,
            "config": {
                "window_samples": 3000, "pre_p_samples": 100,
                "sample_rate_hz": 100, "p_jitter_samples": P_JITTER_SAMPLES,
                "event_disjoint": True, "balanced_sampling": BALANCED_SAMPLING,
            }
        }, BEST)
    else:
        bad_epochs += 1

    if bad_epochs >= PATIENCE:
        print("Early stopping.")
        break

hist = pd.DataFrame(history)
hist.to_csv(OUTPUT_DIR/"training_history.csv", index=False)
print("Best validation MAE:", best_mae)
print("Checkpoint:", BEST)

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(hist.epoch, hist.train_mae, marker="o", ms=3, label="train MAE")
ax.plot(hist.epoch, hist.val_mae, marker="o", ms=3, label="validation MAE")
ax.set(xlabel="Epoch", ylabel="Magnitude MAE", title="EQViT-TXED learning curve")
ax.legend()
ax.grid(alpha=.2)
plt.show()

## 10. Locked test-set verification

Only now do we load the best validation checkpoint and evaluate the held-out earthquake set.

In [ ]:
ckpt = torch.load(BEST, map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"])
te = run_epoch(model, test_loader, training=False)

print(f"Best epoch : {ckpt['epoch']}")
print(f"Test MAE   : {te['mae']:.4f}")
print(f"Test RMSE  : {te['rmse']:.4f}")
print(f"Test bias  : {te['bias']:.4f}")
print(f"Residual σ : {te['std']:.4f}")

pred_df = test_df.copy().reset_index(drop=True)
assert len(pred_df) == len(te["pred"])
pred_df["predicted_magnitude"] = te["pred"]
pred_df["residual"] = pred_df.predicted_magnitude - pred_df.magnitude
pred_df["abs_error"] = pred_df.residual.abs()
pred_df.to_csv(OUTPUT_DIR/"test_predictions.csv", index=False)
display(pred_df.head())

In [ ]:
lo = min(pred_df.magnitude.min(), pred_df.predicted_magnitude.min())
hi = max(pred_df.magnitude.max(), pred_df.predicted_magnitude.max())

fig, ax = plt.subplots(figsize=(6,6))
ax.scatter(pred_df.magnitude, pred_df.predicted_magnitude, s=8, alpha=.25)
ax.plot([lo,hi],[lo,hi],"--",lw=1)
ax.set(xlabel="Catalog magnitude", ylabel="Predicted magnitude",
       title=f"Event-disjoint TXED test | MAE={te['mae']:.3f}")
ax.grid(alpha=.2)
plt.show()

fig, ax = plt.subplots(figsize=(8,4))
ax.hist(pred_df.residual, bins=60)
ax.axvline(0, ls="--", lw=1)
ax.set(xlabel="Predicted − catalog magnitude", ylabel="Count",
       title="Test residual distribution")
ax.grid(alpha=.2)
plt.show()

## 11. Performance by magnitude — especially M≥3 and M≥4

Aggregate MAE can hide the behavior most relevant to EEW. Report large-event subsets explicitly whenever enough examples exist.

In [ ]:
def subset_metrics(d):
    if len(d) == 0:
        return {"N":0, "MAE":np.nan, "RMSE":np.nan, "bias":np.nan}
    e = d.residual.values
    return {
        "N": len(d),
        "MAE": np.mean(np.abs(e)),
        "RMSE": np.sqrt(np.mean(e**2)),
        "bias": np.mean(e),
    }

rows = []
for label, q in [
    ("all", np.ones(len(pred_df), dtype=bool)),
    ("M<1", pred_df.magnitude < 1),
    ("1≤M<2", (pred_df.magnitude >= 1)&(pred_df.magnitude < 2)),
    ("2≤M<3", (pred_df.magnitude >= 2)&(pred_df.magnitude < 3)),
    ("M≥3", pred_df.magnitude >= 3),
    ("M≥4", pred_df.magnitude >= 4),
]:
    rows.append({"subset":label, **subset_metrics(pred_df[q])})
mag_metrics = pd.DataFrame(rows)
display(mag_metrics)
mag_metrics.to_csv(OUTPUT_DIR/"metrics_by_magnitude.csv", index=False)

In [ ]:
bins = np.arange(math.floor(pred_df.magnitude.min()*2)/2,
                 math.ceil(pred_df.magnitude.max()*2)/2 + .51, .5)
pred_df["mag_bin"] = pd.cut(pred_df.magnitude, bins=bins, include_lowest=True)
by_mag = pred_df.groupby("mag_bin", observed=True).agg(
    N=("abs_error","size"),
    MAE=("abs_error","mean"),
    bias=("residual","mean"),
).reset_index()

fig, ax = plt.subplots(figsize=(9,4))
x = np.arange(len(by_mag))
ax.plot(x, by_mag.MAE, marker="o")
ax.set_xticks(x)
ax.set_xticklabels(by_mag.mag_bin.astype(str), rotation=45, ha="right")
ax.set(xlabel="Catalog magnitude bin", ylabel="MAE",
       title="Magnitude-dependent generalization error")
ax.grid(alpha=.2)
plt.tight_layout()
plt.show()

## 12. SNR dependence

The EQViT paper reports that magnitude errors grow at low SNR. This plot checks whether the TXED-trained Torch model exhibits the same expected stress behavior.

In [ ]:
snr_df = pred_df[np.isfinite(pred_df.snr_mean)].copy()
if len(snr_df):
    qs = np.unique(np.nanquantile(snr_df.snr_mean, [0,.2,.4,.6,.8,1]))
    if len(qs) >= 3:
        snr_df["snr_bin"] = pd.cut(snr_df.snr_mean, bins=qs, include_lowest=True, duplicates="drop")
        by_snr = snr_df.groupby("snr_bin", observed=True).agg(
            N=("abs_error","size"), MAE=("abs_error","mean"), bias=("residual","mean")
        ).reset_index()
        display(by_snr)

        fig, ax = plt.subplots(figsize=(8,4))
        ax.scatter(snr_df.snr_mean, snr_df.abs_error, s=7, alpha=.15)
        ax.set(xlabel="Mean 3-C SNR (dB)", ylabel="Absolute magnitude error",
               title="EQViT-TXED error versus SNR")
        ax.grid(alpha=.2)
        plt.show()

## 13. Station generalization diagnostic

A station with unusually high error may indicate response/scaling differences, data-quality issues, or insufficient training coverage.

In [ ]:
station_stats = pred_df.groupby("station").agg(
    N=("abs_error","size"),
    MAE=("abs_error","mean"),
    bias=("residual","mean"),
    Mmean=("magnitude","mean"),
).query("N >= 10").sort_values("MAE", ascending=False)

display(station_stats.head(20))
station_stats.to_csv(OUTPUT_DIR/"metrics_by_station.csv")

top = station_stats.head(20).sort_values("MAE")
fig, ax = plt.subplots(figsize=(8,6))
ax.barh(top.index, top.MAE)
ax.set(xlabel="MAE", ylabel="Station",
       title="Highest-error stations (N≥10)")
ax.grid(axis="x", alpha=.2)
plt.show()

## 14. P-pick-error stress test

Operationally, the magnitude model will be centered on an **EQCCT-torch P pick**, not a perfect analyst pick. We therefore deliberately shift P and measure degradation.

A robust model should degrade gradually rather than fail abruptly.

In [ ]:
@torch.no_grad()
def evaluate_fixed_shift(frame, shift_samples):
    ds = TXEDMagnitudeDataset(H5_PATH, frame, jitter_samples=0, seed=SEED)

    # Override item extraction only for this diagnostic.
    ys, ps = [], []
    with h5py.File(H5_PATH, "r") as f:
        for start in range(0, len(frame), BATCH_SIZE*2):
            chunk = frame.iloc[start:start+BATCH_SIZE*2]
            xx, yy = [], []
            for r in chunk.itertuples():
                g = f[r.id]
                x = np.asarray(g["data"], np.float32)
                p = int(round(float(g.attrs["p_arrival_sample"]))) + shift_samples
                w = extract_window(x, p, 3000, 100)
                w = w - w.mean(axis=0, keepdims=True)
                xx.append(w); yy.append(float(g.attrs["magnitude"]))
            xb = torch.from_numpy(np.stack(xx)).to(DEVICE)
            pred = model(xb).reshape(-1).cpu().numpy()
            ys.extend(yy); ps.extend(pred)
    y = np.asarray(ys); p = np.asarray(ps)
    return np.mean(np.abs(p-y))

stress_frame = test_df if not QUICK_RUN else test_df.iloc[:min(1000,len(test_df))]
stress = []
for shift in [-100,-50,-20,0,20,50,100]:
    mae = evaluate_fixed_shift(stress_frame, shift)
    stress.append((shift, shift/100., mae))
    print(f"shift={shift:+4d} samples ({shift/100:+.2f}s): MAE={mae:.4f}")

stress = pd.DataFrame(stress, columns=["shift_samples","shift_seconds","MAE"])
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(stress.shift_seconds, stress.MAE, marker="o")
ax.set(xlabel="P-pick error (s)", ylabel="Magnitude MAE",
       title="Robustness to automatic P-pick timing error")
ax.grid(alpha=.2)
plt.show()

## 15. Inspect the worst predictions

This is often more useful than another aggregate metric. Look for clipping, gaps, abnormal station response, low SNR, incorrect P metadata, or unusual waveforms.

In [ ]:
worst = pred_df.nlargest(8, "abs_error")
display(worst[["id","station","magnitude","predicted_magnitude","residual","snr_mean"]])

with h5py.File(H5_PATH, "r") as f:
    for r in worst.head(4).itertuples():
        g = f[r.id]
        x = np.asarray(g["data"], np.float32)
        p = int(round(float(g.attrs["p_arrival_sample"])))
        w = extract_window(x,p)
        w = w-w.mean(0,keepdims=True)
        t=np.arange(3000)/100-1
        fig,ax=plt.subplots(figsize=(10,3.5))
        sc=np.max(np.abs(w),axis=0); sc[sc==0]=1
        for j,lab in enumerate(["Z","N/1","E/2"]):
            ax.plot(t,w[:,j]/sc[j]+2*j,lw=.7,label=lab)
        ax.axvline(0,ls="--",lw=1)
        ax.set(title=f"{r.id} | catalog={r.magnitude:.2f}, predicted={r.predicted_magnitude:.2f}, "
                     f"error={r.residual:+.2f}, SNR={r.snr_mean:.1f} dB",
               xlabel="Time relative to P (s)", ylabel="Display-normalized traces")
        ax.legend(ncol=3)
        ax.grid(alpha=.15)
        plt.show()

## 16. Event-level verification

The network is a single-station estimator, but an operational TexNet EEW system can combine multiple station estimates for the same event. Here we aggregate held-out station predictions by event using the median, which is robust to a bad station.

In [ ]:
event_pred = pred_df.groupby("event_id").agg(
    catalog_magnitude=("magnitude","median"),
    predicted_magnitude=("predicted_magnitude","median"),
    n_stations=("station","nunique"),
).reset_index()
event_pred["residual"] = event_pred.predicted_magnitude-event_pred.catalog_magnitude
event_pred["abs_error"] = event_pred.residual.abs()

print("Held-out events:", len(event_pred))
print("Event-level median-fusion MAE:", event_pred.abs_error.mean())
print("Event-level RMSE:", np.sqrt(np.mean(event_pred.residual**2)))

fig, ax = plt.subplots(figsize=(6,6))
lo=min(event_pred.catalog_magnitude.min(),event_pred.predicted_magnitude.min())
hi=max(event_pred.catalog_magnitude.max(),event_pred.predicted_magnitude.max())
ax.scatter(event_pred.catalog_magnitude,event_pred.predicted_magnitude,
           s=10,alpha=.35)
ax.plot([lo,hi],[lo,hi],"--",lw=1)
ax.set(xlabel="Catalog magnitude",ylabel="Median station prediction",
       title="Held-out event-level magnitude")
ax.grid(alpha=.2)
plt.show()

## 17. Save a compact verification report

The files in `outputs/txed_eqvit_magnitude/` are enough to compare later models (manual-P, EQCCT-P, 30-s, 10-s, 4-s, etc.) without rerunning this notebook.

In [ ]:
report = {
    "checkpoint": str(BEST),
    "best_epoch": int(ckpt["epoch"]),
    "best_validation_mae": float(ckpt["val_mae"]),
    "test": {k: float(te[k]) for k in ["mae","rmse","bias","std"]},
    "n_test_waveforms": int(len(pred_df)),
    "n_test_events": int(pred_df.event_id.nunique()),
    "event_level_mae_median_fusion": float(event_pred.abs_error.mean()),
    "config": ckpt.get("config", {}),
}
with open(OUTPUT_DIR/"verification_summary.json","w") as f:
    json.dump(report,f,indent=2)

print(json.dumps(report,indent=2))
print("\nSaved:")
for p in sorted(OUTPUT_DIR.glob("*")):
    print(" ",p)

# What to do next

Once this notebook runs successfully in full mode:

1. Replace manual-P-centered training with **EQCCT-torch predicted P samples** and repeat the same locked test.
2. Compare `P_JITTER_SAMPLES = 0, 20, 50, 100`.
3. Train separate magnitude horizons (for example P+4 s, P+10 s, P+20 s, P+29 s) to obtain a true **EEW latency–accuracy curve**.
4. Evaluate station-held-out and time-held-out splits in addition to the event-held-out split.
5. Add response/instrument-aware experiments if station-dependent amplitude scaling is found to dominate residuals.

The current notebook is intentionally the **robust 30-s TXED baseline** against which those later EEW experiments should be compared.